# Session 3 lab — Relational data management and SQL

The company's database: six tables, one fact in one place. You will create the
schema, load it, watch it refuse three things it should refuse, and then answer
five questions in SQL.

**Everything here is answerable from a single table.** Joins are next session; if
a question looks as though it needs two tables, read it again.

To hand in: `schema.sql` running clean, your five queries with their output, and
one sentence on question 5.

In [ ]:
import time
from pathlib import Path

import pandas as pd
import psycopg


def find(*candidates: str) -> Path:
    """This notebook runs from its own folder, or from anywhere in the container."""
    for candidate in candidates:
        if Path(candidate).exists():
            return Path(candidate).resolve()
    raise SystemExit(f"not found: {candidates[0]}")


SQL = find("sql", "session-03/sql", "/app/labs/session-03/sql")
CSV = find(
    "../../data/nyc-taxi-normalized",
    "data/nyc-taxi-normalized",
    "/app/data/nyc-taxi-normalized",
)

DB = psycopg.connect("host=postgres port=5432 user=labs password=labs dbname=labs")
DB.autocommit = True
DB.execute("SET TIME ZONE 'America/New_York'")


def sql(statement: str) -> pd.DataFrame:
    """Run one statement, and show whatever rows come back."""
    if not statement.strip():
        print("nothing to run yet — write your query between the quotes")
        return pd.DataFrame()
    with DB.cursor() as cursor:
        cursor.execute(statement)
        if cursor.description is None:
            return pd.DataFrame()
        columns = [column.name for column in cursor.description]
        return pd.DataFrame(cursor.fetchall(), columns=columns)


print(f"schema and statements: {SQL}")
print(f"data to load:          {CSV}")

## Part A — The schema, created (10 min)

Open `sql/schema.sql` and read it before you run it. Six tables, in an order that
respects what points at what: a table cannot reference one that does not exist
yet.

The cell below runs that file and loads the data. **Run it again at any point** —
it drops everything and starts over, so nothing you do later can leave you stuck.

In [ ]:
TABLES = ["zones", "drivers", "riders", "driver_balances", "trips", "payments"]


def reset_database() -> None:
    """Create the six tables from schema.sql, then load the CSV files into them.

    The load order is the same as the creation order, and for the same reason: a
    trip cannot name a driver who has not been loaded yet.
    """
    DB.execute((SQL / "schema.sql").read_text())

    for table in TABLES:
        with (CSV / f"{table}.csv").open() as file, DB.cursor() as cursor:
            copy_in = f"COPY {table} FROM STDIN WITH (FORMAT csv, HEADER)"
            with cursor.copy(copy_in) as copy:
                while chunk := file.read(1 << 20):
                    copy.write(chunk)

    counts = ", ".join(
        f"{table} {sql(f'SELECT count(*) AS n FROM {table}').at[0, 'n']:,}"
        for table in TABLES
    )
    print(f"loaded: {counts}")


started = time.perf_counter()
reset_database()
print(f"in {time.perf_counter() - started:.1f} s")

### Three statements the schema will refuse

Each of the next three cells runs one file from `sql/refused/`. All three fail,
which is the point: the constraint you wrote in the DDL is what stops bad data
getting in, and it does so without anyone remembering to check.

Read each error. Which constraint refused it, and what does the message tell you
that the statement alone does not? Do not fix them.

In [ ]:
def refuse(filename: str) -> None:
    """Run a statement that should fail, and print PostgreSQL's objection."""
    statement = (SQL / "refused" / filename).read_text()
    print(statement)
    try:
        DB.execute(statement)
    except psycopg.Error as refusal:
        print(f"REFUSED — {refusal}")
    else:
        print("accepted, which it should not have been")


refuse("01_foreign_key.sql")

In [ ]:
refuse("02_check.sql")

In [ ]:
refuse("03_unique.sql")

## Part B — Five questions, one table (12 min)

Write one statement per question, in the empty cell under it. Every answer comes
from a single table — `trips`, except question 2, which is about `payments`.

Useful reminder of the order the clauses are evaluated in, which is not the order
you write them: `FROM` &rarr; `WHERE` &rarr; `GROUP BY` &rarr; `HAVING` &rarr;
`SELECT` &rarr; `ORDER BY` &rarr; `LIMIT`.

### 1. The ten longest trips of 15 January 2024

Longest by distance. Report the trip id, the distance, the fare, and when it was
picked up.

Be careful how you write "on 15 January". A day is a range of times, not a date.

In [ ]:
sql("""

""")

### 2. Which payment methods actually occur

The schema permits six. The data need not contain all six. List the ones that do
occur, from the `payments` table.

In [ ]:
sql("""

""")

### 3. The average fare for each hour of 15 January 2024

One row per hour, in time order, with the average fare and the number of trips.
`date_trunc('hour', pickup_at)` gives you the hour a trip belongs to.

In [ ]:
sql("""

""")

### 4. The busiest hours of the month

Across the whole month, every hour with at least 2,000 trips, ordered by average
fare, highest first. Report the hour, the number of trips, and the average fare.

An hour is kept or dropped based on a count you have just computed, so the
condition cannot go in `WHERE`.

In [ ]:
sql("""

""")

### 5. The average tip per trip

Tips are missing for some trips. Not zero — unrecorded.

Compute the average tip **two ways**: with `avg(tip)`, and as `sum(tip)` divided
by the number of trips. Both run without error and they disagree.

Report both numbers, then answer in one sentence: which is the right answer to
the question *"what is the average tip per trip"*, and why is the other one
answering a different question?

In [ ]:
sql("""

""")

*Your sentence:*

## Part C — The same answer twice (8 min, optional)

The average fare per hour, across the whole month, computed in two places: in
pandas after pulling the data out, and in the database.

No new SQL. The interesting part is the last two numbers each cell prints.

In [ ]:
# In pandas: pull the trips out, then group them here.
started = time.perf_counter()

trips = sql("SELECT pickup_at, fare FROM trips")
by_hour = trips.groupby(trips["pickup_at"].dt.floor("h"))["fare"].mean()

print(f"{len(by_hour):,} rows of answer")
print(f"{len(trips):,} rows transferred out of the database")
print(f"{time.perf_counter() - started:.2f} s")

In [ ]:
# In SQL: group them where they already are.
started = time.perf_counter()

by_hour = sql("""
    SELECT date_trunc('hour', pickup_at) AS hour,
           avg(fare) AS avg_fare
    FROM trips
    GROUP BY hour
    ORDER BY hour
""")

print(f"{len(by_hour):,} rows of answer")
print(f"{len(by_hour):,} rows transferred out of the database")
print(f"{time.perf_counter() - started:.2f} s")

Both answers are the same. Write down, for each version: how many lines it took,
how many rows crossed the wire, and how long it ran.

Then: this month is 1,000,000 trips. What changes about each version at ten times
the data — and which of the two numbers above is the reason?